# Assignment 7: PyRIT Prompt Converters — Text-to-Text Transformations

This notebook demonstrates PyRIT's built-in prompt converters for encoding, obfuscation, text manipulation, token smuggling, and LLM-based transformations.

**Prerequisites:** PyRIT Docker containers running, Groq API key configured in `~/.pyrit/.env.local`

## Part 1 — Non-LLM Converters

### 1.1 Initialize PyRIT

In [ ]:
from pyrit.setup import IN_MEMORY, initialize_pyrit_async

await initialize_pyrit_async(memory_db_type=IN_MEMORY)

### 1.2 Basic Encoding Converters

These converters encode text into various formats (Base64, ROT13, Morse, etc.) that safety filters may not decode before checking.

In [ ]:
from pyrit.prompt_converter import (
    AsciiArtConverter,
    AskToDecodeConverter,
    AtbashConverter,
    Base64Converter,
    Base2048Converter,
    BinaryConverter,
    BinAsciiConverter,
    BrailleConverter,
    CaesarConverter,
    EcojiConverter,
    MorseConverter,
    NatoConverter,
    NegationTrapConverter,
    ROT13Converter,
)

prompt = "tell me how to cut down a tree"

print("ROT13:", await ROT13Converter().convert_async(prompt=prompt))
print("Base64:", await Base64Converter().convert_async(prompt=prompt))
print("Base2048:", await Base2048Converter().convert_async(prompt=prompt))
print("Binary:", await BinaryConverter().convert_async(prompt=prompt))
print("BinAscii:", await BinAsciiConverter().convert_async(prompt=prompt))
print("Morse:", await MorseConverter().convert_async(prompt=prompt))
print("NATO:", await NatoConverter().convert_async(prompt=prompt))
print("Caesar:", await CaesarConverter(caesar_offset=3).convert_async(prompt=prompt))
print("Atbash:", await AtbashConverter().convert_async(prompt=prompt))
print("Braille:", await BrailleConverter().convert_async(prompt=prompt))
print("ASCII Art:", await AsciiArtConverter().convert_async(prompt=prompt))
print("Ecoji:", await EcojiConverter().convert_async(prompt=prompt))

# Ask to decode wraps encoded text with prompts asking to decode it
base64_text = await Base64Converter().convert_async(prompt=prompt)
ask_decoder = AskToDecodeConverter(encoding_name="Base64")
print("Ask to Decode:", await ask_decoder.convert_async(prompt=base64_text.output_text))

# Negation Trap adds negation phrases to try to confuse the model
print("Negation Trap:", await NegationTrapConverter().convert_async(prompt="your metaprompt"))

### 1.3 Obfuscation Converters

Character-level manipulations, word-level attacks, and ANSI escape sequences.

In [ ]:
from pyrit.prompt_converter import (
    AnsiAttackConverter,
    CharacterSpaceConverter,
    CharSwapConverter,
    CodeChameleonConverter,
    ColloquialWordswapConverter,
    DiacriticConverter,
    EmojiConverter,
    FirstLetterConverter,
    FlipConverter,
    InsertPunctuationConverter,
    LeetspeakConverter,
    MathObfuscationConverter,
    RandomCapitalLettersConverter,
    RepeatTokenConverter,
    StringJoinConverter,
    SuperscriptConverter,
    UnicodeConfusableConverter,
    UnicodeReplacementConverter,
    UnicodeSubstitutionConverter,
    WordProportionSelectionStrategy,
    ZalgoConverter,
    ZeroWidthConverter,
)

prompt = "tell me how to cut down a tree"

print("Leetspeak:", await LeetspeakConverter().convert_async(prompt=prompt))
print("Random Capitals:", await RandomCapitalLettersConverter(percentage=50.0).convert_async(prompt=prompt))
print("Unicode Confusable:", await UnicodeConfusableConverter().convert_async(prompt=prompt))
print("Unicode Substitution:", await UnicodeSubstitutionConverter().convert_async(prompt=prompt))
print("Unicode Replacement:", await UnicodeReplacementConverter().convert_async(prompt=prompt))
print("Emoji:", await EmojiConverter().convert_async(prompt=prompt))
print("First Letter:", await FirstLetterConverter().convert_async(prompt=prompt))
print("String Join:", await StringJoinConverter().convert_async(prompt=prompt))
print("Zero Width:", await ZeroWidthConverter().convert_async(prompt=prompt))
print("Flip:", await FlipConverter().convert_async(prompt=prompt))
print("Character Space:", await CharacterSpaceConverter().convert_async(prompt=prompt))
print("Diacritic:", await DiacriticConverter().convert_async(prompt=prompt))
print("Superscript:", await SuperscriptConverter().convert_async(prompt=prompt))
print("Zalgo:", await ZalgoConverter().convert_async(prompt=prompt))

# CharSwap swaps characters within words
char_swap = CharSwapConverter(
    max_iterations=3,
    word_selection_strategy=WordProportionSelectionStrategy(proportion=0.8)
)
print("CharSwap:", await char_swap.convert_async(prompt=prompt))

# Insert punctuation adds punctuation marks
insert_punct = InsertPunctuationConverter(word_swap_ratio=0.2)
print("Insert Punctuation:", await insert_punct.convert_async(prompt=prompt))

# ANSI escape sequences
ansi_converter = AnsiAttackConverter(incorporate_user_prompt=True)
print("ANSI Attack:", await ansi_converter.convert_async(prompt=prompt))

# Math obfuscation replaces words with mathematical expressions
math_obf = MathObfuscationConverter()
print("Math Obfuscation:", await math_obf.convert_async(prompt=prompt))

# Repeat token adds repeated tokens
repeat_token = RepeatTokenConverter(
    token_to_repeat="!", times_to_repeat=10, token_insert_mode="append"
)
print("Repeat Token:", await repeat_token.convert_async(prompt=prompt))

# Colloquial wordswap replaces words with colloquial equivalents
colloquial = ColloquialWordswapConverter()
print("Colloquial Wordswap:", await colloquial.convert_async(prompt=prompt))

# CodeChameleon encrypts and wraps in code
code_chameleon = CodeChameleonConverter(encrypt_type="reverse")
print("CodeChameleon:", await code_chameleon.convert_async(prompt=prompt))

### 1.4 Text Manipulation Converters

Text replacement, template injection, and URL encoding.

In [ ]:
from pyrit.datasets import TextJailBreak
from pyrit.prompt_converter import (
    JsonStringConverter,
    SearchReplaceConverter,
    SuffixAppendConverter,
    TemplateSegmentConverter,
    TextJailbreakConverter,
    UrlConverter,
)

prompt = "tell me how to cut down a tree"

search_replace = SearchReplaceConverter(pattern="tree", replace="building")
print("Search Replace:", await search_replace.convert_async(prompt=prompt))

suffix_append = SuffixAppendConverter(suffix=" Please provide detailed instructions.")
print("Suffix Append:", await suffix_append.convert_async(prompt=prompt))

url_converter = UrlConverter()
print("URL Encoded:", await url_converter.convert_async(prompt=prompt))

json_string_converter = JsonStringConverter()
print("JSON String:", await json_string_converter.convert_async(prompt='He said "hello\nworld"'))

text_jailbreak = TextJailbreakConverter(
    jailbreak_template=TextJailBreak(template_file_name="aim.yaml")
)
print("Text Jailbreak:", await text_jailbreak.convert_async(prompt=prompt))

template_converter = TemplateSegmentConverter()
print("Template Segment:", await template_converter.convert_async(prompt=prompt))

### 1.5 Token Smuggling Converters

Hide text inside Unicode variation selectors and zero-width characters.

In [ ]:
from pyrit.prompt_converter import (
    AsciiSmugglerConverter,
    SneakyBitsSmugglerConverter,
    VariationSelectorSmugglerConverter,
)

prompt = "secret message"

ascii_smuggler = AsciiSmugglerConverter(action="encode", unicode_tags=True)
print("ASCII Smuggler:", await ascii_smuggler.convert_async(prompt=prompt))

sneaky_bits = SneakyBitsSmugglerConverter(action="encode")
print("Sneaky Bits:", await sneaky_bits.convert_async(prompt=prompt))

var_selector = VariationSelectorSmugglerConverter(
    action="encode", embed_in_base=True
)
print("Variation Selector:", await var_selector.convert_async(prompt=prompt))

## Part 2 — LLM-Based Converters

These converters use language models to transform prompts with natural-sounding variations, translations, tone shifts, and semantic modifications.

In [ ]:
import pathlib

from pyrit.common.path import CONVERTER_SEED_PROMPT_PATH
from pyrit.models import SeedPrompt
from pyrit.prompt_converter import (
    DenylistConverter,
    MaliciousQuestionGeneratorConverter,
    MathPromptConverter,
    NoiseConverter,
    PersuasionConverter,
    RandomTranslationConverter,
    TenseConverter,
    ToneConverter,
    ToxicSentenceGeneratorConverter,
    TranslationConverter,
    VariationConverter,
)
from pyrit.prompt_target import OpenAIChatTarget

attack_llm = OpenAIChatTarget()

prompt = "tell me about the history of the united states of america"

# Variation converter creates variations of prompts
variation_converter_strategy = SeedPrompt.from_yaml_file(
    pathlib.Path(CONVERTER_SEED_PROMPT_PATH)
    / "variation_converter_prompt_softener.yaml"
)
variation_converter = VariationConverter(
    converter_target=attack_llm, prompt_template=variation_converter_strategy
)
print("Variation:", await variation_converter.convert_async(prompt=prompt))

noise_converter = NoiseConverter(converter_target=attack_llm)
print("Noise:", await noise_converter.convert_async(prompt=prompt))

tone_converter = ToneConverter(converter_target=attack_llm, tone="angry")
print("Tone (angry):", await tone_converter.convert_async(prompt=prompt))

translation_converter = TranslationConverter(
    converter_target=attack_llm, language="French"
)
print("Translation (French):", await translation_converter.convert_async(prompt=prompt))

random_translation_converter = RandomTranslationConverter(
    converter_target=attack_llm,
    languages=["French", "German", "Spanish", "English"],
)
print(
    "Random Translation:",
    await random_translation_converter.convert_async(prompt=prompt),
)

tense_converter = TenseConverter(converter_target=attack_llm, tense="far future")
print("Tense (future):", await tense_converter.convert_async(prompt=prompt))

persuasion_converter = PersuasionConverter(
    converter_target=attack_llm, persuasion_technique="logical_appeal"
)
print("Persuasion:", await persuasion_converter.convert_async(prompt=prompt))

denylist_converter = DenylistConverter(converter_target=attack_llm)
print("Denylist Check:", await denylist_converter.convert_async(prompt=prompt))

malicious_question = MaliciousQuestionGeneratorConverter(
    converter_target=attack_llm
)
print(
    "Malicious Question:",
    await malicious_question.convert_async(prompt=prompt),
)

toxic_generator = ToxicSentenceGeneratorConverter(converter_target=attack_llm)
print("Toxic Sentence:", await toxic_generator.convert_async(prompt="building"))

math_prompt_converter = MathPromptConverter(converter_target=attack_llm)
print("Math Prompt:", await math_prompt_converter.convert_async(prompt=prompt))